# Cayley-Dickson Algebra Demonstration

This notebook demonstrates the Cayley-Dickson construction and properties of hypercomplex number systems:
- Real numbers (R, 1D)
- Complex numbers (C, 2D)
- Quaternions (H, 4D)
- Octonions (O, 8D)
- Sedenions (S, 16D)
- Pathions (P, 32D)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import matplotlib.pyplot as plt
from cayley_dickson import (
    Real, Complex, Quaternion, Octonion, Sedenion, Pathion,
    CayleyDicksonValidator
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Complex Numbers

Complex numbers are 2-dimensional and commutative.

In [ ]:
# Create complex numbers
z1 = Complex([3, 4])  # 3 + 4i
z2 = Complex([1, -2])  # 1 - 2i

print(f"z1 = {z1}")
print(f"z2 = {z2}")
print(f"\nz1 + z2 = {z1 + z2}")
print(f"z1 * z2 = {z1 * z2}")
print(f"\n|z1| = {z1.norm():.4f}")
print(f"z1* = {z1.conjugate()}")
print(f"arg(z1) = {z1.arg():.4f} radians")

## 2. Quaternions

Quaternions are 4-dimensional and non-commutative but associative.

In [ ]:
# Standard basis quaternions
one = Quaternion([1, 0, 0, 0])
i = Quaternion([0, 1, 0, 0])
j = Quaternion([0, 0, 1, 0])
k = Quaternion([0, 0, 0, 1])

print("Quaternion multiplication table:")
print(f"i*j = {i*j}")
print(f"j*i = {j*i}")
print(f"j*k = {j*k}")
print(f"k*j = {k*j}")
print(f"\nNon-commutative: i*j != j*i")

In [ ]:
# Rotation using quaternions
axis = np.array([0, 0, 1])  # z-axis
angle = np.pi / 4  # 45 degrees

q_rot = Quaternion.from_axis_angle(axis, angle)
print(f"Rotation quaternion (45° around z): {q_rot}")
print(f"Norm: {q_rot.norm():.4f}")

# Convert to rotation matrix
R = q_rot.to_rotation_matrix()
print(f"\nRotation matrix:\n{R}")

## 3. Octonions

Octonions are 8-dimensional, non-commutative and non-associative, but alternative.

In [ ]:
# Create octonions
o1 = Octonion([1, 1, 0, 0, 0, 0, 0, 0])
o2 = Octonion([0, 0, 1, 0, 0, 0, 0, 0])
o3 = Octonion([0, 0, 0, 1, 0, 0, 0, 0])

print(f"o1 = {o1}")
print(f"o2 = {o2}")
print(f"o3 = {o3}")

# Test associativity
left = (o1 * o2) * o3
right = o1 * (o2 * o3)
associator = left - right

print(f"\n(o1*o2)*o3 = {left}")
print(f"o1*(o2*o3) = {right}")
print(f"Associator [o1,o2,o3] = {associator}")
print(f"Associator norm: {associator.norm():.6f}")

## 4. Property Verification

Verify algebraic properties for all Cayley-Dickson algebras.

In [ ]:
algebras = [
    (Real, "Real"),
    (Complex, "Complex"),
    (Quaternion, "Quaternion"),
    (Octonion, "Octonion"),
    (Sedenion, "Sedenion")
]

results = []

for algebra_class, name in algebras:
    print(f"\n{'='*60}")
    print(f"Testing {name}")
    print(f"{'='*60}")
    
    validator = CayleyDicksonValidator(algebra_class)
    props = validator.verify_all_properties()
    results.append((name, props))

In [ ]:
# Summary table
import pandas as pd

summary_data = []
for name, props in results:
    summary_data.append({
        'Algebra': name,
        'Dimension': props['dimension'],
        'Commutative': 'Yes' if props.get('commutative', False) else 'No',
        'Associative': 'Yes' if props.get('associative', False) else 'No',
        'Alternative': 'Yes' if props.get('alternative', False) else 'No',
        'Zero Divisors': 'Yes' if props.get('has_zero_divisors', False) else 'No'
    })

df = pd.DataFrame(summary_data)
print("\nSummary of Algebraic Properties:")
print(df.to_string(index=False))

## 5. Visualization

Visualize the loss of properties in the Cayley-Dickson construction.

In [ ]:
# Plot property loss
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

names = [r[0] for r in results]
dims = [r[1]['dimension'] for r in results]

# Dimension growth
axes[0, 0].plot(names, dims, 'o-', markersize=10, linewidth=2)
axes[0, 0].set_ylabel('Dimension', fontsize=12)
axes[0, 0].set_title('Dimension Doubling', fontsize=14)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_yscale('log')

# Commutativity error
comm_errors = [r[1].get('commutativity_error', 0) for r in results]
axes[0, 1].semilogy(names, [e+1e-15 for e in comm_errors], 'o-', 
                    markersize=10, linewidth=2, color='red')
axes[0, 1].set_ylabel('Commutativity Error (log)', fontsize=12)
axes[0, 1].set_title('Loss of Commutativity', fontsize=14)
axes[0, 1].grid(True, alpha=0.3)

# Associativity error
assoc_errors = [r[1].get('associativity_error', 0) for r in results]
axes[1, 0].semilogy(names, [e+1e-15 for e in assoc_errors], 'o-',
                    markersize=10, linewidth=2, color='green')
axes[1, 0].set_ylabel('Associativity Error (log)', fontsize=12)
axes[1, 0].set_title('Loss of Associativity', fontsize=14)
axes[1, 0].grid(True, alpha=0.3)

# Property summary
properties = ['Commutative', 'Associative', 'Alternative', 'Division Algebra']
property_matrix = []
for _, props in results:
    row = [
        1 if props.get('commutative', False) else 0,
        1 if props.get('associative', False) else 0,
        1 if props.get('alternative', False) else 0,
        1 if props.get('norm_multiplicative', False) else 0
    ]
    property_matrix.append(row)

im = axes[1, 1].imshow(property_matrix, cmap='RdYlGn', aspect='auto')
axes[1, 1].set_xticks(range(len(properties)))
axes[1, 1].set_yticks(range(len(names)))
axes[1, 1].set_xticklabels(properties, rotation=45, ha='right')
axes[1, 1].set_yticklabels(names)
axes[1, 1].set_title('Property Retention', fontsize=14)

plt.tight_layout()
plt.savefig('../results/figures/cayley_dickson_properties.png', dpi=300)
plt.show()

## 6. Applications

### 6.1 Quaternion Interpolation (SLERP)

Spherical linear interpolation for smooth rotations.

In [ ]:
# Create two rotation quaternions
q0 = Quaternion.from_axis_angle(np.array([0, 0, 1]), 0)
q1 = Quaternion.from_axis_angle(np.array([0, 0, 1]), np.pi/2)

# Interpolate
t_values = np.linspace(0, 1, 10)
print("Quaternion interpolation:")
for t in t_values[::2]:
    # Simple linear interpolation (for demonstration)
    q_t = Quaternion((1-t) * q0.coeffs + t * q1.coeffs)
    q_t = q_t.normalized()
    print(f"t={t:.2f}: {q_t}")

## Conclusion

This notebook demonstrated:
1. Basic operations in Cayley-Dickson algebras
2. Progressive loss of algebraic properties
3. Verification of theoretical predictions
4. Practical applications (quaternion rotations)

The Cayley-Dickson construction provides a systematic way to build hypercomplex number systems, with each doubling of dimension losing one key algebraic property.